# PRÉ-PROJETO DE PESQUISA OPERACIONAL — Pyomo com HiGHS, CBC ou Gurobi (passos de 1 minuto)

## Otimização Operacional do Agendamento de Bombas e Distribuição de Água Tratada via Programação Linear Inteira Mista (MILP)

- Escolha o solver alterando apenas a variável `SOLVER` na célula de configuração.
- A célula inicial instala Pyomo, HiGHS, CBC e a interface Python do Gurobi.
- O horizonte é de 24 horas, discretizado em 1.440 intervalos de 1 minuto.

In [ ]:
%pip install --upgrade "pyomo>=6.8" "highspy>=1.8" cbcbox gurobipy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pyomo.environ import (
    ConcreteModel, Set, Var, Param, Objective, Constraint, Binary,
    NonNegativeReals, RangeSet, minimize, value, SolverFactory, TerminationCondition
)
import pyomo.environ as pyo

# 1. Conjuntos de Índices

T = 24
horas = range(T)
MINUTOS_POR_DIA = T * 60
DURACAO_PASSO_MIN = 1
minutos = range(MINUTOS_POR_DIA)
bombas = range(1, 7)
reservatorios = range(1, 6)
setores = range(1, 13)

# 2. Topologia do Sistema (Arcos permitidos conforme o diagrama)
# Arcos (Bomba -> Reservatório)
arcos_PR = [
    (1,1), 
    (2,1), (2,2), (2,3), 
    (3,2), 
    (4,3), (4,4), 
    (5,5), 
    (6,4), (6,5)
]

# Arcos (Reservatório -> Setor de Consumo)
arcos_RD = [
    (1,1), (1,2), (1,3),
    (2,2), (2,4), (2,5),
    (3,4), (3,6), (3,7),
    (4,4), (4,8), (4,9), (4,10), (4,12),
    (5,7), (5,10), (5,11), (5,12)
]

# 3. Parâmetros Econômicos (Tarifa Horossazonal - R$/kWh)
# Tarifa fora de ponta: R$ 0.4525 | Tarifa de ponta (18h00 às 20h59 ): R$ 1.4048
C = {t: 1.4048 if 18 <= t < 21 else 0.4525 for t in horas}

# Defina False para pular o cálculo do cenário-base e seus gráficos.
GERAR_CENARIO_BASE = True

# 4. Parâmetros das bombas: (potência de referência em kW, vazão mínima e máxima em m³/h)
# Potências estimadas no ponto de vazão máxima, com 30 m de altura manométrica e 80% de eficiência global.
# O objetivo considera essa potência constante durante cada período ligado.
param_bombas = {
    1: (240, 1170, 2340),
    2: (450, 2180, 4370),
    3: (240, 1170, 2340),
    4: (290, 1400, 2810),
    5: (190,  940, 1870),
    6: (240, 1170, 2340)
}
Q_ETA_max = 9400  # m³/h; limite de vazão agregada da ETA

# 5. Parâmetros dos Reservatórios: (volume mínimo, máximo e inicial), em m³
# O volume mínimo (15%) é a reserva de segurança; o volume inicial define o estoque no início do horizonte (30%).
param_res = {
    1: (2100, 14000, 4200),
    2: (1400,  9400, 2800),
    3: (1400,  9400, 2800),
    4: (1400,  9400, 2800),
    5: ( 700,  4700, 1400)
}

# 6. Demandas-base dos setores (m³/h); total: 5195 m³/h
base_demanda = {
    1: 312,  2: 468,  3: 390,  4: 624,
    5: 390,  6: 312,  7: 546,  8: 468,
    9: 312, 10: 702, 11: 390, 12: 281
}

# Curva horária de demanda de verão; cada fator multiplica a demanda-base dos setores.
# Demanda média diária: 8312 m³/h; pico: 17663 m³/h às 18h.
curva_verao = [
    0.8, 0.5, 0.3, 0.4, 0.6, 1.0,   # 00h às 05h: menor consumo na madrugada
    1.8, 2.0, 1.9, 1.7, 1.5, 1.5,   # 06h às 11h: pico da manhã
    1.4, 1.5, 1.6, 1.7, 1.8, 2.9,   # 12h às 17h: consumo alto e crescente
    3.4, 3.0, 2.6, 2.0, 1.5, 1.0    # 18h às 23h: pico máximo às 18h
]

D = {}
for d in setores:
    for t in horas:
        D[d, t] = round(base_demanda[d] * curva_verao[t], 2)

# Vazões e demandas por intervalo de 1 minuto, em m³/min.
D_min = {(d, t): D[d, t // 60] / 60 for d in setores for t in minutos}

# Indicadores de demanda e armazenamento calculados para o horizonte diário.
demanda_horaria_total = [sum(D[d, h] for d in setores) for h in horas]
volume_demanda_diario = sum(demanda_horaria_total)
demanda_media_horaria = volume_demanda_diario / T
hora_pico = max(horas, key=lambda h: demanda_horaria_total[h])
demanda_pico_horaria = demanda_horaria_total[hora_pico]
armazenamento_util_total = sum(
    param_res[r][1] - param_res[r][0] for r in reservatorios
)

# Buffer agregado teórico com bombeamento constante igual à demanda média.
saldo_acumulado = [0]
for demanda in demanda_horaria_total:
    saldo_acumulado.append(
        saldo_acumulado[-1] + demanda_media_horaria - demanda
    )
armazenamento_util_estimado = max(saldo_acumulado) - min(saldo_acumulado)

print(f"Soma das demandas-base: {sum(base_demanda.values())} m³/h")
print(f"Demanda média horária: {demanda_media_horaria:.2f} m³/h")
print(
    f"Pico horário: {demanda_pico_horaria:.2f} m³/h às {hora_pico:02d}h"
)
print(f"Volume de demanda diário: {volume_demanda_diario:.2f} m³/dia")
print(f"Volume útil agregado dos reservatórios: {armazenamento_util_total:.0f} m³")
print(
    "Armazenamento útil estimado pelo perfil: "
    f"{armazenamento_util_estimado:.2f} m³ "
    "(estimativa agregada, com bombeamento constante na demanda média)"
)



In [ ]:
# Altere somente esta variável: "highs", "cbc" ou "gurobi".
SOLVER = "highs"

SOLVERS = {
    "highs": {
            "factory": "appsi_highs",
            "options": {
                "time_limit": 6000,
                "mip_rel_gap": 0.003,
                "threads": 0,
                "random_seed": 0,
                "mip_detect_symmetry": True,
                "mip_heuristic_effort": 0.10,
                "mip_heuristic_run_shifting": True,
                "parallel": "choose",                
            },
        },
    "highs_base": {
            "factory": "appsi_highs",
            "options": {
                "time_limit": 3600,
                "mip_rel_gap": 0.003,
                "threads": 0,
                "random_seed": 0,                                
            },
        },    
    "cbc": {
        "factory": "cbc",
        "options": {
            "seconds": 3600,
            "ratio": 0.003,
            "threads": 0,
            "randomSeed": 0,
        },
    },
    "gurobi": {
        "factory": "gurobi_direct",
        "options": {
            "TimeLimit": 300,
            "MIPGap": 0.003,
            "Threads": 0,
            "Seed": 0,
        },
    },
}

if SOLVER not in SOLVERS:
    raise ValueError(
        f"SOLVER inválido: {SOLVER!r}. Escolha entre {', '.join(SOLVERS)}."
    )

def criar_solver(verbose=False):
    config = SOLVERS[SOLVER]
    if SOLVER == "cbc":
        import cbcbox
        solver = SolverFactory(
            config["factory"], executable=cbcbox.cbc_bin_path()
        )
    else:
        solver = SolverFactory(config["factory"])
    if not solver.available(exception_flag=False):
        raise RuntimeError(
            f"O solver {SOLVER!r} não está disponível. Instale/configure o solver "
            "selecionado e, para Gurobi, verifique também a licença."
        )

    for option, setting in config["options"].items():
        solver.options[option] = setting

    if SOLVER in ("highs", "highs_base"):
        solver.options["output_flag"] = verbose
    elif SOLVER == "cbc":
        solver.options["logLevel"] = 1 if verbose else 0
    else:
        solver.options["OutputFlag"] = int(verbose)
    return solver


In [ ]:
# Criação do modelo em Pyomo
model = ConcreteModel(name="Agendamento_Bombas_MILP_MINUTOS")

# Conjuntos
model.P = RangeSet(1, 6)
model.R = RangeSet(1, 5)
model.D = RangeSet(1, 12)
model.T = RangeSet(0, MINUTOS_POR_DIA - 1)
model.ARCO_PR = Set(dimen=2, initialize=arcos_PR)
model.ARCO_RD = Set(dimen=2, initialize=arcos_RD)

# q, f_pr e f_rd são vazões em m³/min; V é volume em m³; x indica acionamento.
model.x = Var(model.P, model.T, domain=Binary)
model.q = Var(model.P, model.T, domain=NonNegativeReals)
model.f_pr = Var(model.ARCO_PR, model.T, domain=NonNegativeReals)
model.f_rd = Var(model.ARCO_RD, model.T, domain=NonNegativeReals)
model.V = Var(model.R, model.T, domain=NonNegativeReals)

# Objetivo: minimizar o custo diário de energia
model.obj = Objective(
    expr=sum(C[t // 60] * param_bombas[p][0] * model.x[p, t] / 60
             for p in model.P for t in model.T),
    sense=minimize
)

# Limites operacionais das bombas
model.limite_min_bomba = Constraint(
    model.P, model.T,
    rule=lambda m, p, t: m.q[p, t] >= param_bombas[p][1] / 60 * m.x[p, t]
)
model.limite_max_bomba = Constraint(
    model.P, model.T,
    rule=lambda m, p, t: m.q[p, t] <= param_bombas[p][2] / 60 * m.x[p, t]
)

def regra_fluxo_bomba(m, p, t):
    arcos_saida = [arco for arco in arcos_PR if arco[0] == p]
    return m.q[p, t] == sum(m.f_pr[p, r, t] for p, r in arcos_saida)
model.fluxo_bomba = Constraint(model.P, model.T, rule=regra_fluxo_bomba)

# Capacidade máxima da ETA
model.capacidade_ETA = Constraint(
    model.T,
    rule=lambda m, t: sum(m.q[p, t] for p in model.P) <= Q_ETA_max / 60
)

# Atendimento da demanda
model.demanda = Constraint(
    model.D, model.T,
    rule=lambda m, d, t: sum(m.f_rd[r, d, t]
                              for r, setor in arcos_RD if setor == d) == D_min[d, t]
)

# Balanço hídrico dos reservatórios
model.balanco = Constraint(
    model.R, model.T,
    rule=lambda m, r, t: m.V[r, t] ==
        (param_res[r][2] if t == 0 else m.V[r, t - 1])
        + DURACAO_PASSO_MIN * sum(m.f_pr[p, r, t] for p, rr in arcos_PR if rr == r)
        - DURACAO_PASSO_MIN * sum(m.f_rd[r, d, t] for rr, d in arcos_RD if rr == r)
)

model.V_min = Constraint(
    model.R, model.T,
    rule=lambda m, r, t: m.V[r, t] >= param_res[r][0]
)
model.V_max = Constraint(
    model.R, model.T,
    rule=lambda m, r, t: m.V[r, t] <= param_res[r][1]
)
model.recuperacao = Constraint(
    model.R,
    rule=lambda m, r: m.V[r, MINUTOS_POR_DIA - 1] >= param_res[r][2]
)


## Cenário-base com tarifa fixa (opcional)

Defina `GERAR_CENARIO_BASE` na célula de parâmetros como `True` para executar o cenário de referência. Ele mantém as mesmas demandas, capacidades e balanços dos reservatórios, mas usa uma tarifa uniforme na função objetivo; na ponta, fixa o bombeamento total na demanda agregada horária, limitado à capacidade da ETA. O custo é então recalculado com as tarifas horárias reais. Quando desativado, a comparação e os gráficos do cenário-base são omitidos, sem afetar a otimização tarifária. Se não houver solução viável para o cenário-base, a otimização tarifária também prossegue normalmente.

In [ ]:
# O cenário-base é opcional; a otimização tarifária não depende dele.
modelo_base = None
resultado_base = None
status_base = None
consumo_base_kwh = None
custo_base_real = None
vazao_base_ponta = None
cenario_base_disponivel = False

if GERAR_CENARIO_BASE:
    # Clona o modelo para calcular uma programação sem sinal de preço horário.
    # Cada intervalo representa 1 minuto: energia = potência (kW) / 60.
    modelo_base = model.clone()
    modelo_base.obj.set_value(
        sum(param_bombas[p][0] * modelo_base.x[p, t] / 60
            for p in modelo_base.P for t in modelo_base.T)
    )

    # Na ponta, mantém em cada minuto a vazão horária agregada da demanda,
    # limitada à capacidade da ETA.
    horas_ponta = [h for h in horas if C[h] == max(C.values())]
    minutos_ponta = [t for t in minutos if t // 60 in horas_ponta]
    vazao_base_ponta = {
        h: min(sum(D[d, h] for d in setores), Q_ETA_max)
        for h in horas_ponta
    }
    modelo_base.vazao_ponta = Constraint(
        minutos_ponta,
        rule=lambda m, t: sum(m.q[p, t] for p in m.P) == vazao_base_ponta[t // 60] / 60,
    )

    solver_base = criar_solver(verbose=False)
    resultado_base = solver_base.solve(modelo_base)
    status_base = resultado_base.solver.termination_condition
    consumo_base_kwh = value(modelo_base.obj, exception=False)

    if consumo_base_kwh is None:
        print(
            "O cenário-base não encontrou solução viável. "
            f"Status: {status_base}. Comparação e gráficos do cenário-base serão ignorados."
        )
    else:
        cenario_base_disponivel = True
        # Recalcula o custo com as tarifas reais sem alterar a programação-base.
        custo_base_real = sum(
            C[t // 60] * param_bombas[p][0] * value(modelo_base.x[p, t]) / 60
            for p in modelo_base.P for t in modelo_base.T
        )
        print("[CENÁRIO-BASE: TARIFA FIXA]")
        print("Status:", status_base)
        print("Vazão-alvo na ponta (m³/h):", vazao_base_ponta)
        print(f"Consumo de energia: {consumo_base_kwh:.2f} kWh")
        print(f"Custo dessa programação com as tarifas reais: R$ {custo_base_real:.2f}")
else:
    print("Cálculo do cenário-base desativado; a otimização tarifária continuará normalmente.")


In [ ]:
# Otimização com o solver selecionado na variável SOLVER
ACOMPANHAR_PROGRESSO = True
INTERVALO_ACOMPANHAMENTO_MIN = 1
historico_progresso = []

solver = criar_solver(verbose=True)

# tee=True mostra o log do solver selecionado
import time
tempo_inicio = time.perf_counter()
acompanhar_com_highs = ACOMPANHAR_PROGRESSO and SOLVER in ("highs", "highs_base")
if acompanhar_com_highs:
    if INTERVALO_ACOMPANHAMENTO_MIN not in (1, 5):
        raise ValueError("INTERVALO_ACOMPANHAMENTO_MIN deve ser 1 ou 5.")

    intervalo_amostragem_s = INTERVALO_ACOMPANHAMENTO_MIN * 60
    solver.options["mip_min_logging_interval"] = min(5, intervalo_amostragem_s)
    solver.set_instance(model)
    highs_model = solver._solver_model
    if highs_model is None or not hasattr(highs_model, "cbMipLogging"):
        raise RuntimeError(
            "A versão do highspy não disponibiliza o callback de progresso do HiGHS."
        )

    estado_amostragem = {"proxima_amostra_s": 0.0}

    def numero_finito(valor):
        if valor is None:
            return None
        numero = float(valor)
        return numero if np.isfinite(numero) else None

    def registrar_progresso(evento, *args):
        dados = evento.data_out if hasattr(evento, "data_out") else args[1]
        tempo_amostra_s = numero_finito(dados.running_time)
        if tempo_amostra_s is None or tempo_amostra_s < estado_amostragem["proxima_amostra_s"]:
            return

        custo = numero_finito(dados.mip_primal_bound)
        limite = numero_finito(dados.mip_dual_bound)
        gap = numero_finito(dados.mip_gap)
        nos_amostra = numero_finito(dados.mip_node_count)
        historico_progresso.append({
            "Tempo (min)": tempo_amostra_s / 60,
            "Melhor custo viável (R$)": custo,
            "Limite inferior (R$)": limite,
            "Gap (%)": gap * 100 if gap is not None else None,
            "Nós": int(nos_amostra) if nos_amostra is not None and nos_amostra >= 0 else None,
        })
        estado_amostragem["proxima_amostra_s"] = (
            int(tempo_amostra_s // intervalo_amostragem_s) + 1
        ) * intervalo_amostragem_s

    highs_model.cbMipLogging += registrar_progresso
    try:
        resultado = solver.solve(model, tee=True)
    finally:
        highs_model.cbMipLogging -= registrar_progresso
else:
    if ACOMPANHAR_PROGRESSO and SOLVER not in ("highs", "highs_base"):
        print(
            f"Amostragem do progresso não está disponível para {SOLVER.upper()}; "
            "a tabela mostrará apenas o resultado final."
        )
    resultado = solver.solve(model, tee=True)
tempo_execucao = time.perf_counter() - tempo_inicio
status_terminal = resultado.solver.termination_condition
solucao_disponivel = status_terminal in (
    TerminationCondition.optimal,
    TerminationCondition.feasible,
    TerminationCondition.maxTimeLimit,
)

limite_inferior = resultado.problem.lower_bound
incumbente = resultado.problem.upper_bound
if incumbente is None and solucao_disponivel:
    incumbente = value(model.obj)
if limite_inferior is not None and incumbente is not None and abs(incumbente) > 1e-12:
    gap_final = abs(incumbente - limite_inferior) / abs(incumbente)
else:
    gap_final = None

nos = None
try:
    tempo = float(resultado.solver.wallclock_time)
except Exception:
    tempo = tempo_execucao
print(f"\n[RESULTADOS DO {SOLVER.upper()} VIA PYOMO]")
print("Status:", status_terminal)
print(f"Custo/solução incumbente: R$ {incumbente:.2f}" if incumbente is not None else "Incumbente: não disponível")
print(f"Melhor limite inferior: R$ {limite_inferior:.2f}" if limite_inferior is not None else "Limite inferior: não disponível")
print(f"Gap relativo final: {gap_final:.2%}" if gap_final is not None else "Gap relativo final: não disponível")
print(f"Tempo: {tempo:.2f} s")
try:
    nos_relatorio = resultado.solver.statistics.branch_and_bound.number_of_created_subproblems
    if nos is None and nos_relatorio not in (None, float("inf")):
        nos = int(nos_relatorio)
except Exception:
    pass
if acompanhar_com_highs:
    info = highs_model.getInfo()
    if info.mip_node_count >= 0:
        nos = int(info.mip_node_count)
print(f"Nós explorados: {nos}" if nos is not None else "Nós explorados: não disponível")

if ACOMPANHAR_PROGRESSO:
    linha_final = {
        "Tempo (min)": tempo_execucao / 60,
        "Melhor custo viável (R$)": float(incumbente) if incumbente is not None else None,
        "Limite inferior (R$)": float(limite_inferior) if limite_inferior is not None else None,
        "Gap (%)": gap_final * 100 if gap_final is not None else None,
        "Nós": int(nos) if nos is not None else None,
    }
    if not historico_progresso or historico_progresso[-1]["Tempo (min)"] < linha_final["Tempo (min)"]:
        historico_progresso.append(linha_final)
    else:
        historico_progresso[-1] = linha_final

    tabela_progresso = pd.DataFrame(historico_progresso)
    display(tabela_progresso.round({
        "Tempo (min)": 2,
        "Melhor custo viável (R$)": 2,
        "Limite inferior (R$)": 2,
        "Gap (%)": 3,
    }))

if solucao_disponivel:
    print(f"\n[SUCESSO] Solução encontrada pelo {SOLVER.upper()} via Pyomo!")
else:
    print(f"\nModelo inviável ou sem solução viável encontrada pelo {SOLVER.upper()}.")


In [ ]:
# Compara a otimização com a referência, se o cenário-base foi calculado.
if solucao_disponivel:
    consumo_otimizado_kwh = sum(
        param_bombas[p][0] * value(model.x[p, t]) / 60
        for p in model.P for t in model.T
    )
    custo_otimizado_real = sum(
        C[t // 60] * param_bombas[p][0] * value(model.x[p, t]) / 60
        for p in model.P for t in model.T
    )

    linhas_comparacao = []
    if cenario_base_disponivel:
        linhas_comparacao.append({
            "Cenário": "Base (tarifa fixa)",
            "Energia (kWh)": consumo_base_kwh,
            "Custo com tarifa real (R$)": custo_base_real,
        })
    linhas_comparacao.append({
        "Cenário": "Otimização (tarifa horária)",
        "Energia (kWh)": consumo_otimizado_kwh,
        "Custo com tarifa real (R$)": custo_otimizado_real,
    })
    comparacao_custos = pd.DataFrame(linhas_comparacao)
    display(comparacao_custos)

    if cenario_base_disponivel:
        economia = custo_base_real - custo_otimizado_real
        percentual_economia = 100 * economia / custo_base_real if custo_base_real else 0
        print(f"Diferença de custo: R$ {economia:.2f} ({percentual_economia:.2f}%)")
    else:
        print("Cenário-base indisponível; a comparação de custos foi omitida.")
else:
    print("A otimização tarifária não encontrou uma solução viável para comparar.")


# GRÁFICOS E RESULTADOS

In [ ]:
# Gráficos independentes para cada cenário disponível; passo temporal de 1 minuto.
if solucao_disponivel:
    import matplotlib.pyplot as plt

    tempo_horas = [t / 60 for t in minutos]
    tempo_horas_com_fim = [t / 60 for t in range(MINUTOS_POR_DIA + 1)]
    preco_energia = [C[t // 60] for t in minutos] + [C[23]]
    cores_res = ["#4CAF50", "#8BC34A", "#CDDC39", "#FFEB3B", "#FFC107"]

    def plotar_graficos_cenario(modelo_cenario, nome_cenario):
        vazao_total = [
            60 * sum(value(modelo_cenario.q[p, t]) for p in bombas)
            for t in minutos
        ]

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
        ax1.bar(
            tempo_horas, vazao_total, width=1 / 60, align="edge",
            color="#2196F3", alpha=0.8, edgecolor="none",
            label="Vazão bombeada (m³/h)",
        )
        ax1.set_ylabel("Vazão total (m³/h)", color="#0b5394", fontweight="bold")
        ax1.tick_params(axis="y", labelcolor="#0b5394")
        ax1.set_title(f"Perfil de bombeamento e tarifa — {nome_cenario}", fontsize=14)
        ax1_twin = ax1.twinx()
        ax1_twin.step(
            tempo_horas_com_fim, preco_energia, where="post",
            color="#f44336", linewidth=2, label="Tarifa de energia (R$/kWh)"
        )
        ax1_twin.set_ylabel("Tarifa (R$/kWh)", color="#990000", fontweight="bold")
        ax1_twin.tick_params(axis="y", labelcolor="#990000")
        ax1.axvspan(18, 21, color="red", alpha=0.1)

        for r in reservatorios:
            volume = [param_res[r][2]] + [
                value(modelo_cenario.V[r, t]) for t in minutos
            ]
            ax2.plot(
                tempo_horas_com_fim, volume, linewidth=1.5,
                color=cores_res[r - 1], label=f"Reservatório {r}"
            )
        ax2.axvspan(18, 21, color="red", alpha=0.1)
        ax2.set_xlabel("Hora do dia (0 = início, 24 = final)", fontweight="bold")
        ax2.set_ylabel("Volume armazenado (m³)", fontweight="bold")
        ax2.set_title("Dinâmica de carga e descarga dos reservatórios", fontsize=14)
        ax2.set_xticks(range(0, 25, 2))
        ax2.set_xlim(0, 24)
        ax2.legend(loc="upper left", bbox_to_anchor=(1.05, 1))
        ax1_twin.legend(loc="upper right", bbox_to_anchor=(1.25, 1.15))
        ax2.grid(True, linestyle="--", alpha=0.5)
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(12, 6))
        for r in reservatorios:
            V_max = param_res[r][1]
            volume = [param_res[r][2]] + [
                value(modelo_cenario.V[r, t]) for t in minutos
            ]
            nivel_percentual = [100 * v / V_max for v in volume]
            plt.plot(
                tempo_horas_com_fim, nivel_percentual, linewidth=1.5,
                color=cores_res[r - 1], label=f"Reservatório {r}"
            )
        plt.axvspan(
            18, 21, color="red", alpha=0.1,
            label="Horário de ponta (18h00–20h59)"
        )
        plt.axhline(100, color="black", linestyle=":", alpha=0.5)
        plt.title(f"Nível dos reservatórios (%) — {nome_cenario}", fontsize=14)
        plt.xlabel("Hora do dia (0 = início, 24 = final)", fontweight="bold")
        plt.ylabel("Volume armazenado (%)", fontweight="bold")
        plt.xticks(range(0, 25, 2))
        plt.xlim(0, 24)
        plt.ylim(0, 105)
        plt.grid(True, linestyle="--", alpha=0.7)
        plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
        plt.tight_layout()
        plt.show()

    plotar_graficos_cenario(model, "Otimização tarifária")
    if cenario_base_disponivel:
        plotar_graficos_cenario(modelo_base, "Cenário-base (tarifa fixa)")
else:
    print("Execute a otimização com sucesso antes de gerar os gráficos.")
